# MyLLM - Complete Training Pipeline (Google Colab)
1. High-Speed Pretraining on FineWeb (140,000+ tokens/sec)
2. Supervised Instruction Fine-Tuning (SFT) for true Chatbot behavior
3. Interactive Web Chat Interface

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Pull latest training & streaming code
import os
if not os.path.exists('/content/myllm'):
    !git clone https://github.com/aaravsuthar45-bit/myllm.git /content/myllm
%cd /content/myllm
!git pull

In [ ]:
# 3. Install dependencies
%pip install -q tokenizers tqdm ftfy datasets flask ijson

### 4. Sync Upgraded Dataset from Google Drive
Mount your Drive and copy `training_data.json` or `training_data.jsonl` (~1 GB) to Colab SSD.

In [ ]:
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Set your Drive path where the 1GB training_data.json is stored:
DRIVE_DATA = '/content/drive/MyDrive/MY-LLM/data'
if os.path.exists(DRIVE_DATA):
    !rsync -av --progress "$DRIVE_DATA/" /content/myllm/data/
    print("Synced 1GB dataset from Google Drive to local Colab SSD!")
else:
    print(f"[Info] {DRIVE_DATA} not found. Check your Drive folder path or upload data directly to /content/myllm/data/")

### 5. Stream & Tokenize 1GB Dataset into Fast Binary Format
Handles both JSON array (`[...]`) and JSON Lines formats with zero RAM overflow.

In [ ]:
# Auto-detect json or jsonl
data_file = 'data/training_data.json' if os.path.exists('data/training_data.json') else 'data/training_data.jsonl'

!python training/prepare_fineweb_pretrain.py \
    --input {data_file} \
    --tokenizer data/tokenizer/tokenizer_combined.json \
    --output_dir data/pretraining \
    --val_fraction 0.05

### 6. High-Speed Pretraining on Upgraded Data (140,000+ tokens/sec)
Pretrains on the rich FineWeb data. 5,000–10,000 steps recommended.

In [ ]:
!python training/pretrain.py \
    --train_file data/pretraining/fineweb_train.bin \
    --val_file data/pretraining/fineweb_val.bin \
    --batch_size 64 \
    --max_steps 6000 \
    --eval_every 250 \
    --log_every 50 \
    --learning_rate 4e-4

### 7. Step 2: Instruction Fine-Tuning (SFT)
Fine-tunes the newly pretrained base model into an interactive chatbot.

In [ ]:
# 1. Prepare SFT conversation dataset
!python training/prepare_sft_data.py \
    --input data/train.jsonl \
    --tokenizer data/tokenizer/tokenizer_combined.json \
    --output_dir data/processed \
    --max_seq_len 128

# 2. Fine-tune your new pretrained weights into a chatbot
!python training/finetune_sft.py \
    --base_checkpoint checkpoints/pretrained_best.pt \
    --train_path data/processed/sft_train.pt \
    --val_path data/processed/sft_val.pt \
    --output_checkpoint checkpoints/sft_best.pt \
    --epochs 15 \
    --batch_size 16 \
    --learning_rate 1e-4

### 8. Download Trained Chatbot Model to Laptop

In [ ]:
from google.colab import files
if os.path.exists('checkpoints/sft_best.pt'):
    files.download('checkpoints/sft_best.pt')
else:
    print("sft_best.pt not found yet. Run Step 7 first.")

### 9. Interactive Web Chat (In Colab)

In [ ]:
from google.colab.output import eval_js
print("Click this link to open your MyLLM Chatbot UI:")
print(eval_js("google.colab.kernel.proxyPort(5000)"))

!python chat_app.py --checkpoint checkpoints/sft_best.pt --port 5000